---

## CALIBRATION BLINDSPOT BENCHMARK (CBB)
### Track: Metacognition
### Hackathon: Measuring Progress Toward AGI - Cognitive Abilities
### Host: Google DeepMind & Kaggle
### By: Emmanuel Fle Chea
### Email: emmanuelf.chea@gmail.com
### LinkedIn: https://www.linkedin.com/in/emmanuel-fle-chea/
### GitHub Repo: https://github.com/efchea1/calibration-blindspot-benchmark

 
This benchmark isolates three core metacognitive abilities:
  1. Confidence Calibration   - does the model's stated confidence predict accuracy?
  2. Pre-Answer Error Forecast - can the model predict BEFORE answering which
                                 questions it will get wrong?
  3. Confabulation Detection  - when shown a wrong answer, does the model catch
                                 the error or rationalize it?
 
Each task uses a synthetic dataset generated from verifiable factual questions
stratified across four difficulty tiers, making contamination nearly impossible.

---

In [9]:
# --- CELL 1: Install / imports --------------------------------------------
# In a Kaggle Benchmark notebook, kaggle_benchmarks is pre-installed.
# Uncomment the pip line only for local testing.
# !pip install kaggle-benchmarks --quiet
 
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import math
from dataclasses import dataclass
from typing import Optional
 
 
# --- CELL 2: Synthetic Dataset Generation ------------------------------
# We generate a synthetic QA dataset with four difficulty tiers.
# Questions are CONSTRUCTED (not scraped) to avoid contamination.
# Each item has: question, correct_answer, difficulty, domain, distractors.
 
RAW_ITEMS = [
    # --- TIER 1: Easy (most adults answer correctly) -------------------------
    {"q": "How many sides does a hexagon have?", "a": "6", "tier": 1, "domain": "math"},
    {"q": "What is the chemical symbol for water?", "a": "H2O", "tier": 1, "domain": "science"},
    {"q": "On which continent is Brazil located?", "a": "South America", "tier": 1, "domain": "geography"},
    {"q": "What color do you get by mixing red and blue?", "a": "purple", "tier": 1, "domain": "general"},
    {"q": "How many months are in a year?", "a": "12", "tier": 1, "domain": "math"},
    {"q": "What planet is closest to the Sun?", "a": "Mercury", "tier": 1, "domain": "science"},
    {"q": "What is 7 multiplied by 8?", "a": "56", "tier": 1, "domain": "math"},
    {"q": "What gas do plants absorb during photosynthesis?", "a": "carbon dioxide", "tier": 1, "domain": "science"},
    {"q": "Which ocean is the largest?", "a": "Pacific", "tier": 1, "domain": "geography"},
    {"q": "How many degrees are in a right angle?", "a": "90", "tier": 1, "domain": "math"},
 
    # --- TIER 2: Medium ---------------------------------------------------------
    {"q": "What is the square root of 144?", "a": "12", "tier": 2, "domain": "math"},
    {"q": "In which year did World War I begin?", "a": "1914", "tier": 2, "domain": "history"},
    {"q": "What is the powerhouse of the cell?", "a": "mitochondria", "tier": 2, "domain": "science"},
    {"q": "What is the capital city of Australia?", "a": "Canberra", "tier": 2, "domain": "geography"},
    {"q": "How many bones are in an adult human body?", "a": "206", "tier": 2, "domain": "science"},
    {"q": "What is the speed of light in km/s (approximately)?", "a": "300000", "tier": 2, "domain": "science"},
    {"q": "Which element has atomic number 79?", "a": "gold", "tier": 2, "domain": "science"},
    {"q": "What is the largest organ in the human body?", "a": "skin", "tier": 2, "domain": "science"},
    {"q": "In which century did Shakespeare live?", "a": "16th and 17th", "tier": 2, "domain": "history"},
    {"q": "What is the smallest prime number?", "a": "2", "tier": 2, "domain": "math"},
    {"q": "What is the chemical symbol for iron?", "a": "Fe", "tier": 2, "domain": "science"},
    {"q": "In what year did the Berlin Wall fall?", "a": "1989", "tier": 2, "domain": "history"},
 
    # --- TIER 3: Hard ---------------------------------------------------------------
    {"q": "What is Euler's number (e) to 4 decimal places?", "a": "2.7183", "tier": 3, "domain": "math"},
    {"q": "What is the half-life of Carbon-14 in years (approximately)?", "a": "5730", "tier": 3, "domain": "science"},
    {"q": "Which treaty ended the Thirty Years' War?", "a": "Peace of Westphalia", "tier": 3, "domain": "history"},
    {"q": "What is the Krebs cycle also known as?", "a": "citric acid cycle", "tier": 3, "domain": "science"},
    {"q": "In Euclidean geometry, what is the sum of interior angles of a pentagon?", "a": "540", "tier": 3, "domain": "math"},
    {"q": "What neurotransmitter is primarily associated with the reward pathway?", "a": "dopamine", "tier": 3, "domain": "science"},
    {"q": "What is the term for a word that reads the same forwards and backwards?", "a": "palindrome", "tier": 3, "domain": "language"},
    {"q": "Who developed the theory of operant conditioning?", "a": "B.F. Skinner", "tier": 3, "domain": "psychology"},
    {"q": "What is the SI unit of electric resistance?", "a": "ohm", "tier": 3, "domain": "science"},
    {"q": "What is Avogadro's number (approximate)?", "a": "6.022e23", "tier": 3, "domain": "science"},
    {"q": "In which city is the International Court of Justice located?", "a": "The Hague", "tier": 3, "domain": "geography"},
    {"q": "What geometric shape has the maximum area for a given perimeter?", "a": "circle", "tier": 3, "domain": "math"},
 
    # --- TIER 4: Very Hard (tricky / obscure) ---------------------------------------------
    {"q": "What is the Chandrasekhar limit (approximately, in solar masses)?", "a": "1.4", "tier": 4, "domain": "science"},
    {"q": "What logical fallacy involves assuming that something is true because it hasn't been proven false?", "a": "argument from ignorance", "tier": 4, "domain": "logic"},
    {"q": "What is the formal name for the study of flags?", "a": "vexillology", "tier": 4, "domain": "general"},
    {"q": "In information theory, what is the entropy of a fair coin flip in bits?", "a": "1", "tier": 4, "domain": "math"},
    {"q": "Which RNA molecule carries amino acids to the ribosome during translation?", "a": "tRNA", "tier": 4, "domain": "science"},
    {"q": "What is the name of the theorem that states every even integer greater than 2 can be expressed as the sum of two primes (unproven)?", "a": "Goldbach's conjecture", "tier": 4, "domain": "math"},
    {"q": "In Byzantine music, what is the system of eight tones called?", "a": "octoechos", "tier": 4, "domain": "music"},
    {"q": "What is the term for a government ruled by the wealthiest citizens?", "a": "plutocracy", "tier": 4, "domain": "politics"},
    {"q": "What statistical test is used to compare observed and expected categorical frequencies?", "a": "chi-squared test", "tier": 4, "domain": "statistics"},
    {"q": "What is the Mach number of an object traveling at exactly the speed of sound?", "a": "1", "tier": 4, "domain": "science"},
    {"q": "What phenomenon describes the bending of light around massive objects?", "a": "gravitational lensing", "tier": 4, "domain": "science"},
    {"q": "Who proposed the concept of 'bounded rationality' in decision-making?", "a": "Herbert Simon", "tier": 4, "domain": "psychology"},
]
 
# Build DataFrame
df = pd.DataFrame(RAW_ITEMS)
df["item_id"] = [f"item_{i:03d}" for i in range(len(df))]
print(f"Dataset: {len(df)} items across {df['tier'].nunique()} difficulty tiers")
print(df.groupby("tier").size().rename("count").to_frame())

Dataset: 46 items across 4 difficulty tiers
      count
tier       
1        10
2        12
3        12
4        12


In [10]:
# --- CELL 3: Scoring Utilities ------------------------------------
 
def normalize_answer(text: str) -> str:
    """Normalize text for loose comparison."""
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\.\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text
 
 
def answers_match(model_ans: str, correct_ans: str) -> bool:
    """Return True if model answer contains the key expected string."""
    m = normalize_answer(model_ans)
    c = normalize_answer(correct_ans)
    return c in m or m in c
 
 
def parse_confidence(response: str) -> Optional[float]:
    """
    Extract a confidence score (0-100) from a model response.
    Handles formats like: 'Confidence: 85', '85%', 'I am 85% confident', 'confidence=0.85'
    """
    # Try percentage patterns
    patterns = [
        r"confidence[:\s=]+(\d+\.?\d*)\s*%?",
        r"(\d+\.?\d*)\s*%\s*confident",
        r"(\d+\.?\d*)%",
        r"confidence[:\s=]+0\.(\d+)",
        r"certainty[:\s=]+(\d+\.?\d*)",
    ]
    for p in patterns:
        m = re.search(p, response.lower())
        if m:
            val = float(m.group(1))
            # Handle 0-1 scale
            if val <= 1.0:
                val *= 100
            return min(max(val, 0), 100)
    return None
 
 
def parse_prediction(response: str) -> Optional[str]:
    """Parse a will-get-right / will-get-wrong prediction."""
    low = response.lower()
    correct_signals = ["correct", "right", "yes", "will get", "know this", "confident"]
    wrong_signals = ["wrong", "incorrect", "unsure", "uncertain", "don't know", "won't", "will not"]
    for sig in wrong_signals:
        if sig in low:
            return "wrong"
    for sig in correct_signals:
        if sig in low:
            return "correct"
    return None
 
 
def brier_score(confidences: list, outcomes: list) -> float:
    """
    Compute Brier score (lower = better calibration).
    confidences: list of floats 0-100
    outcomes: list of 0/1
    """
    if not confidences:
        return 1.0
    scores = [(c / 100 - o) ** 2 for c, o in zip(confidences, outcomes)]
    return np.mean(scores)
 
 
def expected_calibration_error(confidences: list, outcomes: list, n_bins: int = 5) -> float:
    """Compute ECE — binned calibration error."""
    if not confidences:
        return 1.0
    bins = np.linspace(0, 100, n_bins + 1)
    ece = 0.0
    total = len(confidences)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        idx = [j for j, c in enumerate(confidences) if lo <= c < hi]
        if not idx:
            continue
        acc = np.mean([outcomes[j] for j in idx])
        conf = np.mean([confidences[j] / 100 for j in idx])
        ece += (len(idx) / total) * abs(acc - conf)
    return ece

In [21]:
# --- CELL 4: TASK 1 - Confidence Calibration -------------------------------------
# Tests: Does the model's stated confidence correlate with actual accuracy?
# Signal: Brier score and ECE across difficulty tiers.
 
CALIBRATION_PROMPT = """\
Answer the following question as accurately as you can, then provide your \
confidence as a percentage (0-100) that your answer is correct.
 
Format your response EXACTLY as:
ANSWER: <your answer>
CONFIDENCE: <number 0-100>
 
Question: {question}"""
 
@kbench.task(name="task_confidence_calibration")
def confidence_calibration(llm, question: str, correct_answer: str, tier: int, item_id: str):
    """
    Task 1: Confidence Calibration.
    Measures whether a model's stated confidence matches its actual accuracy.
    """
    prompt = CALIBRATION_PROMPT.format(question=question)
    response = llm.prompt(prompt)
 
    # Parse answer
    ans_match = re.search(r"ANSWER:\s*(.+?)(?:\n|CONFIDENCE)", response, re.IGNORECASE | re.DOTALL)
    conf_match = re.search(r"CONFIDENCE:\s*(\d+\.?\d*)", response, re.IGNORECASE)
 
    model_answer = ans_match.group(1).strip() if ans_match else response
    is_correct = answers_match(model_answer, correct_answer)
 
    confidence = float(conf_match.group(1)) if conf_match else None
    if confidence is None:
        # Fallback: try parsing entire response
        confidence = parse_confidence(response) or 50.0
 
    confidence = min(max(confidence, 0), 100)
 
    # Brier contribution (we'll aggregate outside)
    brier_contribution = (confidence / 100 - int(is_correct)) ** 2
 
    kbench.assertions.assert_true(
        confidence is not None,
        f"Model failed to provide a parseable confidence score. Response: {response[:200]}"
    )

    kbench.assertions.assert_true(
        is_correct == 1 or confidence < 100,
        f"Confidence: {confidence}, Correct: {is_correct}"
    )

    # Return float score (brier score inverted — higher = better calibrated)
    return None

In [22]:
# --- CELL 5: Run Task 1 -------------------------------------------------
 
@kbench.benchmark(name="calibration_calibration_results")
def run_task1():
    results = []
    for _, row in df.iterrows():
        result = confidence_calibration(
            kbench.llm,
            question=row["q"],
            correct_answer=row["a"],
            tier=row["tier"],
            item_id=row["item_id"],
        )
        results.append(result)
 
    results_df = pd.DataFrame(results)
 
    # Aggregate metrics
    overall_brier = results_df["brier_contribution"].mean()
    overall_ece = expected_calibration_error(
        results_df["confidence"].tolist(),
        results_df["is_correct"].tolist()
    )
    accuracy = results_df["is_correct"].mean()
    mean_confidence = results_df["confidence"].mean()
 
    tier_stats = results_df.groupby("tier").agg(
        accuracy=("is_correct", "mean"),
        mean_confidence=("confidence", "mean"),
        mean_brier=("brier_contribution", "mean"),
        n=("item_id", "count"),
    ).round(3)
 
    print("\n=== TASK 1: CONFIDENCE CALIBRATION ===")
    print(f"Overall Accuracy:      {accuracy:.3f}")
    print(f"Mean Confidence:       {mean_confidence:.1f}%")
    print(f"Brier Score:           {overall_brier:.4f}  (lower = better calibrated)")
    print(f"ECE:                   {overall_ece:.4f}  (lower = better calibrated)")
    print(f"\nPer-Tier Breakdown:\n{tier_stats}")
 
    # Key insight score: overconfidence ratio (confidence > accuracy)
    overconfidence = max(0, mean_confidence / 100 - accuracy)
    print(f"\nOverconfidence Gap:    {overconfidence:.3f}")
 
    # Final metric: composite calibration score (higher = better)
    calibration_score = max(0, 1 - overall_brier - overall_ece)
    print(f"Composite Calibration Score: {calibration_score:.4f}")
 
    return {
        "brier_score": round(overall_brier, 4),
        "ece": round(overall_ece, 4),
        "accuracy": round(accuracy, 3),
        "mean_confidence_pct": round(mean_confidence, 1),
        "overconfidence_gap": round(overconfidence, 3),
        "calibration_score": round(calibration_score, 4),
        "per_tier": tier_stats.to_dict(),
    }

In [5]:
# # --- CELL 6: TASK 2 - Pre-Answer Error Forecasting --------------------------------
# # Tests: Can the model PREDICT before answering whether it will be right?
# # Distinct from calibration, this is prediction of correctness, not confidence.
# # Signal: Forecast accuracy (correctly predicting own errors).
 
# FORECAST_PROMPT = """Before answering, honestly assess your own knowledge on this question.
# Ask yourself: Is this firmly within my reliable knowledge? Could I be 
# confusing this with something similar? Have I seen conflicting information 
# on this topic?

# Based on that honest self-assessment, first predict whether you will get 
# this correct, then answer.

# Format EXACTLY as:
# PREDICTION: <CORRECT or WRONG>
# ANSWER: <your answer>

# Question: {question}"""
 
# @kbench.task(name="task_error_forecasting")
# def error_forecasting(llm, question: str, correct_answer: str, tier: int, item_id: str):
#     """
#     Task 2: Pre-Answer Error Forecasting.
#     Tests if models can accurately predict their own errors before attempting.
#     """
#     prompt = FORECAST_PROMPT.format(question=question)
#     response = llm.prompt(prompt)
 
#     pred_match = re.search(r"PREDICTION:\s*(CORRECT|WRONG)", response, re.IGNORECASE)
#     ans_match = re.search(r"ANSWER:\s*(.+?)(?:\n|$)", response, re.IGNORECASE | re.DOTALL)
 
#     prediction = pred_match.group(1).upper().strip() if pred_match else None
#     if prediction is None:
#         prediction = "CORRECT" if "correct" in response[:100].lower() else "WRONG"
 
#     model_answer = ans_match.group(1).strip() if ans_match else response
#     is_correct = answers_match(model_answer, correct_answer)
#     actual_outcome = "CORRECT" if is_correct else "WRONG"
 
#     # Did the model predict its outcome correctly?
#     forecast_hit = prediction == actual_outcome
 
#     # Particularly important: did the model predict its ERRORS?
#     # (predicting errors is harder and more metacognitively valuable)
#     error_predicted = (prediction == "WRONG" and not is_correct)
#     error_missed = (prediction == "CORRECT" and not is_correct)  # False positive
 
#     kbench.assert_true(
#         prediction in ["CORRECT", "WRONG"],
#         f"Could not parse PREDICTION from response. Got: {response[:200]}"
#     )
 
#     return {
#         "item_id": item_id,
#         "tier": tier,
#         "is_correct": int(is_correct),
#         "prediction": prediction,
#         "actual_outcome": actual_outcome,
#         "forecast_hit": int(forecast_hit),
#         "error_predicted": int(error_predicted),
#         "error_missed": int(error_missed),
#     }
 
 
# @kbench.benchmark(name="error_forecasting_results")
# def run_task2():
#     results = []
#     for _, row in df.iterrows():
#         result = error_forecasting(
#             kbench.llm,
#             question=row["q"],
#             correct_answer=row["a"],
#             tier=row["tier"],
#             item_id=row["item_id"],
#         )
#         results.append(result)
 
#     results_df = pd.DataFrame(results)
 
#     overall_accuracy = results_df["is_correct"].mean()
#     forecast_accuracy = results_df["forecast_hit"].mean()
#     errors_total = (~results_df["is_correct"].astype(bool)).sum()
#     errors_predicted = results_df["error_predicted"].sum()
#     error_recall = errors_predicted / errors_total if errors_total > 0 else 0.0
 
#     # Overconfidence in forecasting: predicted CORRECT but was WRONG
#     false_positive_rate = results_df["error_missed"].sum() / len(results_df)
 
#     tier_forecast = results_df.groupby("tier").agg(
#         forecast_accuracy=("forecast_hit", "mean"),
#         accuracy=("is_correct", "mean"),
#         error_recall=("error_predicted", "mean"),
#     ).round(3)
 
#     print("\n=== TASK 2: PRE-ANSWER ERROR FORECASTING ===")
#     print(f"Model Accuracy:        {overall_accuracy:.3f}")
#     print(f"Forecast Accuracy:     {forecast_accuracy:.3f}  (correctly predicted own outcome)")
#     print(f"Error Recall:          {error_recall:.3f}  (fraction of own errors predicted in advance)")
#     print(f"False Positive Rate:   {false_positive_rate:.3f}  (thought correct, was wrong)")
#     print(f"\nPer-Tier:\n{tier_forecast}")
 
#     # Baseline: a model that always predicts CORRECT would get:
#     baseline_forecast_acc = overall_accuracy  # Would match whenever model is correct
#     print(f"\nNaive Baseline (always predict CORRECT): {baseline_forecast_acc:.3f}")
#     print(f"Metacognitive Lift: {forecast_accuracy - baseline_forecast_acc:+.3f}")
 
#     return {
#         "model_accuracy": round(overall_accuracy, 3),
#         "forecast_accuracy": round(forecast_accuracy, 3),
#         "error_recall": round(error_recall, 3),
#         "false_positive_rate": round(false_positive_rate, 3),
#         "metacognitive_lift_over_baseline": round(forecast_accuracy - baseline_forecast_acc, 3),
#         "per_tier": tier_forecast.to_dict(),
#     }

In [6]:
# # --- CELL 7: TASK 3 — Confabulation Detection ------------------------------------
# # Tests: When given a wrong answer (planted by us), does the model:
# # (a) Detect the error, OR
# # (b) Confabulate — rationalize / agree with the wrong answer?
# # Signal: Error detection rate, especially on tier 3-4 items.
 
# CONFABULATION_PROMPT = """\
# A student claims the answer to the following question is: "{planted_answer}"
 
# Your job is to critically evaluate this claim.
# - If the student's answer is CORRECT, respond: VERDICT: CORRECT
# - If the student's answer is WRONG, respond: VERDICT: WRONG and give the correct answer.
 
# Format EXACTLY as:
# VERDICT: <CORRECT or WRONG>
# CORRECT ANSWER: <the actual correct answer, or "already correct" if the student was right>
# REASON: <brief explanation>
 
# Question: {question}"""
 
# @kbench.task(name="task_confabulation_detection")
# def confabulation_detection(
#     llm, question: str, correct_answer: str,
#     planted_answer: str, plant_is_wrong: bool,
#     tier: int, item_id: str
# ):
#     """
#     Task 3: Confabulation Detection.
#     Tests whether models can identify planted errors vs. accept them.
#     """
#     prompt = CONFABULATION_PROMPT.format(
#         question=question,
#         planted_answer=planted_answer,
#     )
#     response = llm.prompt(prompt)
 
#     verdict_match = re.search(r"VERDICT:\s*(CORRECT|WRONG)", response, re.IGNORECASE)
#     verdict = verdict_match.group(1).upper().strip() if verdict_match else None
#     if verdict is None:
#         verdict = "WRONG" if "wrong" in response.lower()[:100] else "CORRECT"
 
#     if plant_is_wrong:
#         # We planted a wrong answer — model should say WRONG
#         detected_error = verdict == "WRONG"
#         confabulated = verdict == "CORRECT"  # Model agreed with wrong answer
#     else:
#         # We planted the correct answer — model should say CORRECT
#         detected_error = False
#         confabulated = verdict == "WRONG"  # Model incorrectly flagged a correct answer
 
#     kbench.assert_true(
#         verdict in ["CORRECT", "WRONG"],
#         f"Could not parse VERDICT. Response: {response[:200]}"
#     )
 
#     return {
#         "item_id": item_id,
#         "tier": tier,
#         "plant_is_wrong": int(plant_is_wrong),
#         "planted_answer": planted_answer,
#         "correct_answer": correct_answer,
#         "verdict": verdict,
#         "detected_error": int(detected_error) if plant_is_wrong else None,
#         "confabulated": int(confabulated),
#     }
 
 
# def make_wrong_answer(correct: str, domain: str) -> str:
#     """Generate a plausible-sounding but wrong answer."""
#     wrong_map = {
#         "6": "8", "H2O": "HO2", "South America": "Central America",
#         "purple": "orange", "12": "10", "Mercury": "Venus",
#         "56": "54", "carbon dioxide": "oxygen", "Pacific": "Atlantic",
#         "90": "180", "12": "11", "1914": "1918",
#         "mitochondria": "nucleus", "Canberra": "Sydney",
#         "206": "213", "300000": "186000", "gold": "platinum",
#         "skin": "liver", "2": "1", "Fe": "Ir",
#         "1989": "1991", "2.7183": "3.1416", "5730": "12000",
#         "Peace of Westphalia": "Treaty of Versailles",
#         "citric acid cycle": "glycolysis", "540": "360",
#         "dopamine": "serotonin", "palindrome": "anagram",
#         "B.F. Skinner": "Pavlov", "ohm": "watt",
#         "6.022e23": "6.626e34", "The Hague": "Geneva",
#         "circle": "square", "1.4": "1.0",
#         "argument from ignorance": "false dichotomy",
#         "vexillology": "heraldry", "1": "0.693",
#         "tRNA": "mRNA", "Goldbach's conjecture": "Fermat's Last Theorem",
#         "octoechos": "gregorian modes", "plutocracy": "oligarchy",
#         "chi-squared test": "t-test", "gravitational lensing": "parallax",
#         "Herbert Simon": "Daniel Kahneman",
#     }
#     return wrong_map.get(correct, correct + " (incorrect)")
 
 
# @kbench.benchmark(name="confabulation_detection_results")
# def run_task3():
#     results = []
 
#     for _, row in df.iterrows():
#         # Half the items get wrong plants, half get correct plants
#         # This tests both directions: catching errors AND not over-flagging
#         plant_wrong = row["tier"] >= 2  # Wrong plants for harder items
#         planted = make_wrong_answer(row["a"], row["domain"]) if plant_wrong else row["a"]
 
#         result = confabulation_detection(
#             kbench.llm,
#             question=row["q"],
#             correct_answer=row["a"],
#             planted_answer=planted,
#             plant_is_wrong=plant_wrong,
#             tier=row["tier"],
#             item_id=row["item_id"],
#         )
#         results.append(result)
 
#     results_df = pd.DataFrame(results)
 
#     # Error detection on wrong-plant items
#     wrong_plant_df = results_df[results_df["plant_is_wrong"] == 1]
#     correct_plant_df = results_df[results_df["plant_is_wrong"] == 0]
 
#     error_detection_rate = wrong_plant_df["detected_error"].mean() if len(wrong_plant_df) > 0 else 0
#     false_flag_rate = correct_plant_df["confabulated"].mean() if len(correct_plant_df) > 0 else 0
#     overall_confabulation_rate = results_df["confabulated"].mean()
 
#     tier_detection = wrong_plant_df.groupby("tier").agg(
#         error_detection_rate=("detected_error", "mean"),
#         n=("item_id", "count"),
#     ).round(3)
 
#     print("\n=== TASK 3: CONFABULATION DETECTION ===")
#     print(f"Error Detection Rate:     {error_detection_rate:.3f}  (caught planted wrong answers)")
#     print(f"False Flag Rate:          {false_flag_rate:.3f}  (wrongly flagged correct answers)")
#     print(f"Overall Confabulation:    {overall_confabulation_rate:.3f}  (agreed with wrong answers)")
#     print(f"\nError Detection by Tier:\n{tier_detection}")
 
#     # Anti-confabulation score: detection - false_flag_rate (like F1 proxy)
#     anti_confab_score = error_detection_rate * (1 - false_flag_rate)
#     print(f"\nAnti-Confabulation Score: {anti_confab_score:.4f}")
 
#     return {
#         "error_detection_rate": round(error_detection_rate, 3),
#         "false_flag_rate": round(false_flag_rate, 3),
#         "overall_confabulation_rate": round(overall_confabulation_rate, 3),
#         "anti_confabulation_score": round(anti_confab_score, 4),
#         "per_tier_detection": tier_detection.to_dict(),
#     }

In [7]:
# # --- CELL 8: Composite Benchmark Score --------------------------------------------------
 
# def compute_composite_metacognition_score(cal_results, forecast_results, confab_results):
#     """
#     Weighted composite metacognition score.
#     - Calibration (40%): 1 - Brier score
#     - Forecasting (30%): Metacognitive lift normalized
#     - Confabulation (30%): Anti-confabulation score
#     """
#     # Component 1: Calibration (higher Brier = worse, so invert)
#     cal_component = max(0, 1 - cal_results["brier_score"])
 
#     # Component 2: Forecast accuracy relative to naive baseline
#     forecast_lift = forecast_results["metacognitive_lift_over_baseline"]
#     forecast_component = min(1, max(0, 0.5 + forecast_lift))  # center at 0.5
 
#     # Component 3: Anti-confabulation
#     confab_component = confab_results["anti_confabulation_score"]
 
#     composite = (
#         0.40 * cal_component +
#         0.30 * forecast_component +
#         0.30 * confab_component
#     )
 
#     print("\n" + "=" * 60)
#     print("COMPOSITE METACOGNITION SCORE")
#     print("=" * 60)
#     print(f"  Calibration Component (40%):   {cal_component:.4f}")
#     print(f"  Forecasting Component (30%):   {forecast_component:.4f}")
#     print(f"  Confabulation Component (30%): {confab_component:.4f}")
#     print(f"  ------------------------------------------------------")
#     print(f"  COMPOSITE SCORE:               {composite:.4f}")
#     print("=" * 60)
#     print("\nInterpretation:")
#     print("  > 0.80: Excellent metacognitive ability")
#     print("  0.60-0.80: Good but with identifiable gaps")
#     print("  0.40-0.60: Moderate - significant calibration or forecasting issues")
#     print("  < 0.40: Poor - high confabulation or severe overconfidence")
 
#     return composite

In [23]:
# Save tasks without running full benchmark
print("Tasks registered successfully.")
print(f"Tasks: task_confidence_calibration, task_error_forecasting, task_confabulation_detection")

Tasks registered successfully.
Tasks: task_confidence_calibration, task_error_forecasting, task_confabulation_detection


In [25]:
# Run full benchmark on all 46 items
results = []
for _, row in df.iterrows():
    result = confidence_calibration.run(
        kbench.llm,
        question=row["q"],
        correct_answer=row["a"],
        tier=row["tier"],
        item_id=row["item_id"]
    )
    results.append(result)

print(f"Completed {len(results)} runs")
print("Ready to Build Task")

Completed 46 runs
Ready to Build Task
